# Explore the Tools Gateway Stack and Create the Gateway

The `workshop-tools-gateway-stack` was pre-deployed by Workshop Studio. It created the Lambda functions, IAM roles, and EventBridge schedule that make up the governed Path B — but the AgentCore Gateway API resource itself still has to be created, because `AWS::BedrockAgentCore::Gateway` is a run-time resource that fits better in a scripted step than in CloudFormation.

This notebook is the interactive equivalent of the **Explore the Deployed Stack** CLI page. You will:

1. Verify `workshop-tools-gateway-stack` is `CREATE_COMPLETE`
2. Capture the stack outputs (Gateway role, Sync Lambda, interceptor ARNs)
3. Resolve the Cognito user-pool and client IDs from Module 3 exports
4. Create the `tools-gateway` AgentCore Gateway with a CUSTOM_JWT authorizer and both interceptors wired in (idempotent)
5. Wire the new gateway ID into the Sync Lambda's `GATEWAY_ID` environment variable

When you finish, you will have a gateway ready to receive tool targets in notebook 03.

## Step 1: Verify the Stack

Confirm the Tools Gateway stack reached `CREATE_COMPLETE`. If it did not, stop and check the CloudFormation console before continuing.

In [ ]:
import boto3

region = boto3.session.Session().region_name or "us-west-2"
cfn = boto3.client("cloudformation", region_name=region)

STACK_NAME = "workshop-tools-gateway-stack"

resp = cfn.describe_stacks(StackName=STACK_NAME)
status = resp["Stacks"][0]["StackStatus"]
print(f"Stack:  {STACK_NAME}")
print(f"Status: {status}")

if status != "CREATE_COMPLETE":
    raise RuntimeError(
        f"Stack {STACK_NAME} is not CREATE_COMPLETE (status={status}). "
        "Check the CloudFormation console before continuing."
    )

## Step 2: Capture Stack Outputs

The stack exports the ARNs of the Gateway role, the Sync Lambda, and both interceptor Lambdas. You need the three ARNs (role, request interceptor, response interceptor) to create the gateway in Step 4.

In [ ]:
outputs = {o["OutputKey"]: o["OutputValue"] for o in resp["Stacks"][0]["Outputs"]}

GATEWAY_ROLE_ARN = outputs["GatewayRoleArn"]
SYNC_LAMBDA_ARN = outputs["SyncLambdaArn"]
REQUEST_INTERCEPTOR_ARN = outputs["RequestInterceptorArn"]
RESPONSE_INTERCEPTOR_ARN = outputs["ResponseInterceptorArn"]

print(f"Gateway Role:         {GATEWAY_ROLE_ARN}")
print(f"Sync Lambda:          {SYNC_LAMBDA_ARN}")
print(f"Request Interceptor:  {REQUEST_INTERCEPTOR_ARN}")
print(f"Response Interceptor: {RESPONSE_INTERCEPTOR_ARN}")

## Step 3: Resolve Cognito IDs from Module 3 Exports

The gateway's `CUSTOM_JWT` authorizer validates tokens issued by the Module 3 Cognito user pool. You need:

- `POOL_ID` — the user-pool ID
- `M2M_CLIENT_ID` — the machine-to-machine client (used by the Sync Lambda and tests)
- `WEB_CLIENT_ID` — the web/frontend client (used by interactive users)

Both client IDs are added to `allowedClients` on the authorizer so either can mint a valid token.

In [ ]:
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

POOL_ID = exports["workshop-CognitoUserPoolId"]
M2M_CLIENT_ID = exports["workshop-CognitoM2MClientId"]

cognito = boto3.client("cognito-idp", region_name=region)
clients = []
cognito_paginator = cognito.get_paginator("list_user_pool_clients")
for page in cognito_paginator.paginate(UserPoolId=POOL_ID):
    clients.extend(page.get("UserPoolClients", []))
web_clients = [c["ClientId"] for c in clients if c["ClientId"] != M2M_CLIENT_ID]
WEB_CLIENT_ID = web_clients[0] if web_clients else M2M_CLIENT_ID

ALLOWED_CLIENTS = [c for c in (WEB_CLIENT_ID, M2M_CLIENT_ID) if c]
DISCOVERY_URL = (
    f"https://cognito-idp.{region}.amazonaws.com/{POOL_ID}"
    "/.well-known/openid-configuration"
)

print(f"Pool ID:         {POOL_ID}")
print(f"Web Client ID:   {WEB_CLIENT_ID}")
print(f"M2M Client ID:   {M2M_CLIENT_ID}")
print(f"Discovery URL:   {DISCOVERY_URL}")

## Step 4: Create (or Update) the AgentCore Gateway

This is the boto3 equivalent of running `python3 create_gateway.py` from the CLI page. The call is idempotent: if a gateway named `tools-gateway` already exists, the notebook updates its configuration; otherwise it creates a new one.

The gateway is configured with:

- `ProtocolType: MCP` with semantic search enabled
- `AuthorizerType: CUSTOM_JWT` validating tokens from the Module 3 Cognito user pool
- Two interceptors — request (audit + access control) and response (field sanitization + guardrails)

In [ ]:
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)

GATEWAY_NAME = "tools-gateway"

INTERCEPTOR_CONFIGS = [
    {
        "interceptor": {"lambda": {"arn": REQUEST_INTERCEPTOR_ARN}},
        "interceptionPoints": ["REQUEST"],
        "inputConfiguration": {"passRequestHeaders": True},
    },
    {
        "interceptor": {"lambda": {"arn": RESPONSE_INTERCEPTOR_ARN}},
        "interceptionPoints": ["RESPONSE"],
        "inputConfiguration": {"passRequestHeaders": True},
    },
]

AUTHORIZER_CONFIG = {
    "customJWTAuthorizer": {
        "discoveryUrl": DISCOVERY_URL,
        "allowedClients": ALLOWED_CLIENTS,
    }
}

PROTOCOL_CONFIGURATION = {
    "mcp": {
        "supportedVersions": ["2025-03-26"],
        "searchType": "SEMANTIC",
        "instructions": (
            "Tools Gateway for the Agentic AI Landing Zone. "
            "Search for tools by describing what you need."
        ),
    }
}

existing = [
    g for g in agentcore.list_gateways().get("items", [])
    if g["name"] == GATEWAY_NAME
]

if existing:
    GATEWAY_ID = existing[0]["gatewayId"]
    print(f"Gateway '{GATEWAY_NAME}' already exists (id={GATEWAY_ID}).")
    # AWS does not allow changing protocolConfiguration.searchType after
    # a gateway has targets. Probe target count; omit searchType from the
    # UPDATE payload when targets exist so the update succeeds on re-runs.
    target_count = len(
        agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])
    )
    if target_count == 0:
        protocol_cfg = PROTOCOL_CONFIGURATION
    else:
        protocol_cfg = {
            "mcp": {
                k: v for k, v in PROTOCOL_CONFIGURATION["mcp"].items()
                if k != "searchType"
            }
        }
        print(f"  Gateway has {target_count} target(s); keeping existing searchType.")
    # Interceptors are set at create time and persist; update_gateway does not
    # accept interceptorConfigurations, so it is omitted here (passing it raises
    # a parameter-validation error when this cell re-runs against a live gateway).
    agentcore.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        protocolConfiguration=protocol_cfg,
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=AUTHORIZER_CONFIG,
    )
else:
    resp = agentcore.create_gateway(
        name=GATEWAY_NAME,
        description="Tools Gateway for the Agentic AI Landing Zone",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        protocolConfiguration=PROTOCOL_CONFIGURATION,
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=AUTHORIZER_CONFIG,
        interceptorConfigurations=INTERCEPTOR_CONFIGS,
    )
    GATEWAY_ID = resp["gatewayId"]
    print(f"Created gateway '{GATEWAY_NAME}' (id={GATEWAY_ID}).")

GATEWAY_URL = f"https://{GATEWAY_ID}.gateway.bedrock-agentcore.{region}.amazonaws.com/mcp"
print(f"Gateway URL: {GATEWAY_URL}")

# Poll until the gateway reaches READY — create/update are asynchronous
import time as _time
gw_status = "UNKNOWN"
for attempt in range(24):  # up to 2 minutes
    g = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
    gw_status = g.get("status", "UNKNOWN")
    if gw_status == "READY":
        break
    if gw_status.endswith("FAILED"):
        raise RuntimeError(f"Gateway {GATEWAY_ID} entered failure state: {gw_status}")
    print(f"  [{(attempt+1)*5}s] Gateway status: {gw_status}")
    _time.sleep(5)
else:
    raise RuntimeError(
        f"Gateway {GATEWAY_ID} did not reach READY in 2 minutes (last status: {gw_status}). "
        f"Check the AgentCore console before continuing."
    )
print(f"Gateway status: {gw_status}")

## Step 5: Wire the Gateway ID into the Sync Lambda

The Sync Lambda (`agentcore-gateway-sync`) was deployed with an empty `GATEWAY_ID` environment variable because the gateway did not exist yet at stack-deploy time. Now that the gateway is live, set that variable using a **read-merge-update** pattern so no other environment variables are lost.

In [ ]:
lambda_client = boto3.client("lambda", region_name=region)

SYNC_FN_NAME = "agentcore-gateway-sync"

current = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
env_vars = dict(current.get("Environment", {}).get("Variables", {}))
env_vars["GATEWAY_ID"] = GATEWAY_ID

lambda_client.update_function_configuration(
    FunctionName=SYNC_FN_NAME,
    Environment={"Variables": env_vars},
)

waiter = lambda_client.get_waiter("function_updated_v2")
waiter.wait(FunctionName=SYNC_FN_NAME, WaiterConfig={"Delay": 2, "MaxAttempts": 30})

updated = lambda_client.get_function_configuration(FunctionName=SYNC_FN_NAME)
print(f"Sync Lambda {SYNC_FN_NAME} GATEWAY_ID = {updated['Environment']['Variables'].get('GATEWAY_ID', '')}")

## Step 6: Confirm the Gateway

One last check: list gateways and confirm the `tools-gateway` entry is present with status `READY`.

In [ ]:
for g in agentcore.list_gateways().get("items", []):
    marker = ">" if g["name"] == GATEWAY_NAME else " "
    print(f"{marker} {g['name']:30s}  status={g.get('status', '?'):12s}  id={g['gatewayId']}")

## Summary

You now have a functioning AgentCore Gateway named `tools-gateway` with:

- Cognito `CUSTOM_JWT` authentication for both the web and M2M clients
- The request and response interceptor Lambdas attached
- The Sync Lambda's `GATEWAY_ID` populated so scheduled sync runs will find it

The gateway has **no targets** yet. In the next notebook (`03-curate-tools.ipynb`) you will selectively promote the Flights MCP, Hotels MCP, and search-knowledge-base Lambdas into gateway targets.